# NanoGPT (Learn)

In [1]:
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

cuda


In [2]:
import os
import sys
from pathlib import Path

from pathlib import Path

CWD = os.path.realpath(os.getcwd())
PARENT_DIR = os.path.dirname(CWD)
sys.path.append(PARENT_DIR)

DATA_DIR = Path(PARENT_DIR).parent / 'data'

In [3]:
from reader.loader import TextDataset

dataset = TextDataset(DATA_DIR, device=device)

100%|██████████| 5/5 [00:00<00:00, 160.18it/s]


In [ ]:
from src.modules.architecture.ngram_lm import NgramLanguageModel
from reader.preprocess import decode
import torch

BATCH_SIZE = 16
EMBEDDING_SIZE = 64
SEQ_LENGTH = 32
DROPOUT_RATE = 0.2

N_HEADS = 4
N_BLOCKS = 4
LR = 5e-4

EVAL_ITER = 100
EVAL_INTERVAL = 100
EPOCH_SIZE = 5000

torch.manual_seed(1337)

model = NgramLanguageModel(vocab_size=dataset.vocab_size, n_heads=N_HEADS, embedding_size=EMBEDDING_SIZE, seq_length=SEQ_LENGTH, 
                            n_blocks=N_BLOCKS, dropout_rate=DROPOUT_RATE, device=device)
model = model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR)

print("Number of parameters:")
print(sum(p.numel() for p in model.parameters())/1e6, 'M parameters')

Number of parameters:
0.225113 M parameters


In [5]:
@torch.no_grad()
def estimate_loss(x, y, model):
    model.eval()
    losses = torch.zeros(EVAL_ITER)
    for k in range(EVAL_ITER):
        logits, loss = model(x, y)
        losses[k] = loss.item()
    return losses.mean()

In [6]:
for iter in range(EPOCH_SIZE):

    # every once in a while evaluate the loss on train and val sets
    if iter % EVAL_INTERVAL == 0 or iter == EPOCH_SIZE - 1:
        x_train, y_train = dataset.load_train(BATCH_SIZE)
        x_val, y_val = dataset.load_test(BATCH_SIZE)
        train_losses = estimate_loss(x_train, y_train, model)
        val_losses = estimate_loss(x_val, y_val, model)
        print(f"step {iter}: train loss {train_losses:.4f}, val loss {val_losses:.4f}")

    # sample a batch of data
    x_train, y_train = dataset.load_train(BATCH_SIZE)

    # evaluate the loss
    logits, loss = model(x_train, y_train)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

step 0: train loss 4.6213, val loss 4.6498
step 100: train loss 2.4808, val loss 2.3773
step 200: train loss 2.3635, val loss 2.3044
step 300: train loss 2.1728, val loss 2.2971
step 400: train loss 2.0372, val loss 2.0659
step 500: train loss 2.0261, val loss 2.0402
step 600: train loss 1.9757, val loss 2.0756
step 700: train loss 2.0189, val loss 2.0013
step 800: train loss 1.9005, val loss 2.0288
step 900: train loss 1.8878, val loss 1.9961
step 1000: train loss 1.8176, val loss 1.9375
step 1100: train loss 1.8744, val loss 2.0106
step 1200: train loss 1.7922, val loss 1.8459
step 1300: train loss 1.6957, val loss 1.9228
step 1400: train loss 1.7974, val loss 1.9057
step 1500: train loss 1.7467, val loss 1.8528
step 1600: train loss 1.7864, val loss 1.9074
step 1700: train loss 1.6970, val loss 1.8287
step 1800: train loss 1.7532, val loss 2.0008
step 1900: train loss 1.7388, val loss 1.9610
step 2000: train loss 1.7457, val loss 1.9917
step 2100: train loss 1.8025, val loss 1.8500


In [7]:
# generate from the model
context = torch.zeros((1, 1), dtype=torch.long, device=device)
print(decode(dataset.itos, model.generate(context, max_new_tokens=2000)[0].tolist()))



KIt mononororouy'shishousis thananan  istyfoneatoninoe, ise, esifanat on onifoninoinienove,e, bino'fon, ashisen,  esanikitsecun,owoweeinenerobusty  iseinesin on inex,ston'irexins,es inegeeyanin, win aistasaminanian, cy inamionee.oity Pshy,omaseanesen, wsheninoinecen  on, ainieiry wis,
esey,iexeeronepefy adeeseves aniss seceveneievin itidoiein eaden astily anoeashistevefaryidepae ononen ty esesan ininoesevesthan.soo in ieyeashianonin, Pine,
yionan an, y, ionyifovinan coney,ounanifsatony, wanefininin  y wan inovietouataneshienoerefaiomy, t inonismavevnsestan inasheaminanan,  epeaofyaniepeas aniny inasind, mesedofonefong ovanefy  ooeyefun, onsthinon,oney,    anedelyiney inan gievananilaienes, tiniely anes onesoes anunonefishiesefo ofobede'stoiene, w en.sinenony y tinefoneshea W iney, ununoun., mede?eitoushiny  aieshepouan, yin ineineyieanisteanonanusemefy, anes iep,ienfiatesity  teea tinonainatin orinifonusassnan  "sshiney asese    aouenouofieithieponaney    tissen.esessteay-eneionaouan

In [11]:
from reader.preprocess import encode
context = torch.tensor([encode(dataset.stoi, 'slab')], dtype=torch.long, device=device)
print(decode(dataset.itos, model.generate(context, max_new_tokens=2000)[0].tolist()))

slab'"se wouam?—  taburrinyis   aaugon orelaian oinanesesthyin indines tininey, on eashiniesareieinisesefonolan?inestanae  in  anan,ananananiefeanepanefosiden inusis  ty tinen tyinousopepe, onenuenaninexessenes,, ienan., y  cofoovepen, taiselanyouinusess, onenian,y,onun, a inineyfonesed, estoeseananonan.ss,  y, asthy  oneyouafopiyofonanoonouseyobinon, t, on  ounenan  inananenanouso o     toinanyidesmelan—stononouinin  s asmisignay   ananestionalistyisestitieaninacuis    onany,seniesesesebinienadononinine  an, oveslen, ousens ratheeleshewsuie,an sis iaiee, by weaiiaonanacanagonon on omin stany,oneianeshina-in g,iene, teinninefuny Iionasuisseinedin minen  istinean,y,  beuseeaeeanininayily onisplan     aninesespronyoinin'asevinoron  isese eshinains, asenayonasamifane ievininoe anan. aeseny aoierine, e"eseireainininyonneshshinas   anyyyieyeanin, tistaiselonelaan y ananamoninaestojy tinashanovevey ainedescyoueyominedionanona,  tifin   genanesonevon, iusenidevineas, titheshesesery ieseshaly 